# **Classical ML for Authorship Signal in Lyric Poetry (Supplementary Clustering)**

*An interpretability-focused exploratory analysis and authorship classification of poetic texts.*

by Natalia Zelenko

### **Project Overview**

This project is a **classical machine-learning text analysis** focused on identifying **authorial signal in poetic language** using interpretable, surface-level features.

The project includes three stages, each in a separate notebook:

1. EDA and preprocessing design.

2. Supervised multiclass classification.

3. **Supplementary unsupervised analysis.**

The same preprocessing pipeline defined at stage 1, is used for stages 2 and 3.

**This clustering stage is exploratory rather than evaluative**. The goal is not to recover a single “true” grouping of the data, but to examine the structure of the same word-level TF-IDF feature space used in classification.

Two different clustering approaches are used. k-means forces all poems into groups, making it useful for observing broad grouping tendencies. HDBSCAN instead identifies only denser regions and allows ambiguous poems to remain unassigned.

The goal is not agreement between models but comparison between their behaviors. All clustering is performed on the raw word-level TF-IDF representation. Dimensionality reduction, where used, is only used for visualization and not for clustering itself.

## **0. Setup**

### **0.1 Imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.cluster import KMeans
import hdbscan

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


### **0.2 Loading**

In [ ]:
CSV_PATH = "data/fragmented_original_expanded.csv"

In [ ]:
df_raw_orig = pd.read_csv(CSV_PATH)
display(df_raw_orig.head())
display(df_raw_orig.shape)

,fragment_id,poem_id,author,origin,fragment_index,text
0,1,collins_introduction_to_poetry,Billy Collins,original,0,I ask them to take a poem\r\nand hold it up to...
1,2,collins_introduction_to_poetry,Billy Collins,original,1,I say drop a mouse into a poem\r\nand watch hi...
2,3,collins_introduction_to_poetry,Billy Collins,original,2,I want them to waterski\r\nacross the surface ...
3,4,collins_introduction_to_poetry,Billy Collins,original,3,But all they want to do\r\nis tie the poem to ...
4,5,collins_introduction_to_poetry,Billy Collins,original,4,They begin beating it with a hose\r\nto find o...


(414, 6)

In [ ]:
# create a working copy of the data
df_work = df_raw_orig.copy(deep=True)
df_work.head()

,fragment_id,poem_id,author,origin,fragment_index,text
0,1,collins_introduction_to_poetry,Billy Collins,original,0,I ask them to take a poem\r\nand hold it up to...
1,2,collins_introduction_to_poetry,Billy Collins,original,1,I say drop a mouse into a poem\r\nand watch hi...
2,3,collins_introduction_to_poetry,Billy Collins,original,2,I want them to waterski\r\nacross the surface ...
3,4,collins_introduction_to_poetry,Billy Collins,original,3,But all they want to do\r\nis tie the poem to ...
4,5,collins_introduction_to_poetry,Billy Collins,original,4,They begin beating it with a hose\r\nto find o...


## **1. Cleaning and Preprocessing (As Designed in Part 1)**

### **1.1 Text cleaning**

In [ ]:
# minimal text cleaning
def clean_text_minimal(text):

    if not isinstance(text, str):
        return text

    # remove fragment boundary markers if any
    text = re.sub(r'[ \t]*---[ \t]*', ' ', text)

    # normalize horizontal whitespace (spaces and tabs only)
    text = re.sub(r'[ \t]+', ' ', text)

    # strip leading/trailing whitespace while preserving newlines
    return text.strip()

In [ ]:
# apply to original fragments
df_work['text_clean'] = df_work['text'].apply(clean_text_minimal)


In [ ]:
# sanity check
df_work[['text', 'text_clean']].sample(5)


,text,text_clean
366,Such weight and thick pink bulk\r\nSet in deat...,Such weight and thick pink bulk\r\nSet in deat...
69,After I carried the mouse by the tail\r\nto a ...,After I carried the mouse by the tail\r\nto a ...
244,"I understand this life, I am matter, \r\nyour ...","I understand this life, I am matter, \r\nyour ..."
222,"There we were in the vaulted tunnel running,\r...","There we were in the vaulted tunnel running,\r..."
81,Everyone was gone; I was playing\r\nin the dar...,Everyone was gone; I was playing\r\nin the dar...


### **1.2 Tokenization definition**

In [ ]:
# word-level tokenizer:
TOKEN_PATTERN = r"[A-Za-z]+"


### **1.3 Vectorization definition**

In [ ]:
# define TF–IDF vectorizer with chozen settings
tfidf_vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    lowercase=True,
    ngram_range=(1, 1),
    stop_words=None
)

### **1.4 Vectorization**

In [ ]:
# fit TF–IDF on the full corpus and transform it
X_tfidf = tfidf_vectorizer.fit_transform(df_work['text'])

print(X_tfidf.shape)


(414, 3201)


## **2. Structure-Imposing Clustering: k-Means**

**Question:** If we force the feature space into a fixed number of regions, what structure emerges?

This section uses k-means to explore the TF-IDF feature space. By forcing all fragments into a fixed number of clusters, k-means makes broader grouping tendencies easier to observe. The goal is to examine what kinds of regions emerge when hard boundaries are imposed on the space, allowing comparison with patterns observed earlier in supervised classification.

**Choices of k**

For the k-means stage, we **deliberately do not use clustering quality metrics** such as inertia or silhouette score to choose the number of clusters. In a high-dimensional space where overlap is expected, such metrics would shift the focus from exploration toward optimization.

Instead, **the number of clusters is treated as a resolution parameter** controlling how strongly the space is partitioned. We examine a small set of interpretable values: k = 5 as an anchor aligned with the number of authors, k = 3 as a coarser partition, and k = 8 as a finer partition used to inspect possible substructure and cross-author mixing.

### **2.1 Fitting the model**

In [ ]:
# define k values
k_values = [3, 5, 8]

# fit k-means for each k and store cluster labels in df_work
for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=17,
        n_init=10
    )

    labels = kmeans.fit_predict(X_tfidf)
    df_work[f"kmeans_k{k}"] = labels


### **2.2 Cluster composition by author**

**NOTE THAT even though only one random state is demonstrated here, this section was evaluated after running it with several random states.**

In [ ]:
# Choose which k to analyze
k = 3
cluster_col = f"kmeans_k{k}"

# Build counts table: rows = clusters, columns = authors
counts = (
    df_work
    .groupby([cluster_col, "author"])
    .size()
    .unstack(fill_value=0)
)

# Convert counts to proportions within each cluster
proportions = counts.div(counts.sum(axis=1), axis=0)

# Sort clusters by dominance (max author proportion per cluster)
sorted_proportions = proportions.loc[
    proportions.max(axis=1).sort_values(ascending=False).index
]

# Display styled table
(
    sorted_proportions
    .style
    .format("{:.1%}")
    .background_gradient(cmap="Blues", axis=None)
    .set_caption(f"Cluster composition by author (k = {k}, sorted by dominance)")
)


author,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
kmeans_k3,,,,,
2,13.3%,25.0%,13.3%,30.0%,18.3%
1,16.0%,24.0%,20.0%,17.3%,22.7%
0,22.9%,16.8%,22.2%,19.0%,19.0%


In [ ]:
# choose which k to analyze
k = 5
cluster_col = f"kmeans_k{k}"

# build counts table: rows = clusters, columns = authors
counts = (
    df_work
    .groupby([cluster_col, "author"])
    .size()
    .unstack(fill_value=0)
)

# convert counts to proportions within each cluster
proportions = counts.div(counts.sum(axis=1), axis=0)

# sort clusters by dominance (max author proportion per cluster)
sorted_proportions = proportions.loc[
    proportions.max(axis=1).sort_values(ascending=False).index
]

# display styled table
(
    sorted_proportions
    .style
    .format("{:.1%}")
    .background_gradient(cmap="Blues", axis=None)
    .set_caption(f"Cluster composition by author (k = {k}, sorted by dominance)")
)


author,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
kmeans_k5,,,,,
3,13.6%,27.3%,13.6%,9.1%,36.4%
0,16.9%,27.0%,12.4%,30.3%,13.5%
2,13.5%,19.2%,25.0%,26.9%,15.4%
4,24.8%,13.2%,23.1%,15.7%,23.1%
1,22.3%,18.5%,23.1%,16.9%,19.2%


In [ ]:
# choose which k to analyze
k = 8
cluster_col = f"kmeans_k{k}"

# build counts table: rows = clusters, columns = authors
counts = (
    df_work
    .groupby([cluster_col, "author"])
    .size()
    .unstack(fill_value=0)
)

# convert counts to proportions within each cluster
proportions = counts.div(counts.sum(axis=1), axis=0)

# sort clusters by dominance (max author proportion per cluster)
sorted_proportions = proportions.loc[
    proportions.max(axis=1).sort_values(ascending=False).index
]

# display styled table
(
    sorted_proportions
    .style
    .format("{:.1%}")
    .background_gradient(cmap="Blues", axis=None)
    .set_caption(f"Cluster composition by author (k = {k}, sorted by dominance)")
)


author,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
kmeans_k8,,,,,
2,25.9%,14.8%,14.8%,44.4%,0.0%
1,11.8%,2.9%,32.4%,11.8%,41.2%
6,14.3%,17.1%,14.3%,34.3%,20.0%
0,15.2%,21.2%,21.2%,33.3%,9.1%
5,14.3%,25.0%,7.1%,32.1%,21.4%
7,25.9%,16.7%,27.8%,13.0%,16.7%
3,21.9%,22.9%,19.8%,10.4%,25.0%
4,22.4%,22.4%,20.6%,17.8%,16.8%


Examining cluster composition across three resolutions (k = 3, 5, and 8) **does not reveal clean author-specific partitions in the TF-IDF feature space.**

At a coarse resolution (k = 3), clusters function as broad regions rather than author-aligned groups: all authors are distributed across clusters, with only mild preferences that vary across random states.

Increasing the resolution to k = 8 does not uncover stable sub-author structure. While some runs produce clusters with locally high author proportions, these patterns are inconsistent across random states, suggesting sensitivity to initialization rather than robust stylistic cores.

**The most interpretable behavior occurs at k = 5, though even here clusters do not map cleanly to individual authors.** Instead, they show moderate contrasts without strong dominance, and these contrasts vary across runs.

**Overall, k-means supports a view of the TF-IDF feature space as overlapping and weakly structured.** Apparent cluster-author alignments depend on resolution and initialization, reinforcing the interpretation that stylistic signal in this space is diffuse and relative.


### **2.3 Poem-Level Cluster Coherence**

As the analytical units are fragments derived from full poems, we examine poem-level cluster coherence to assess whether clustering primarily reflects within-poem cohesion or cuts across poem boundaries.

In [ ]:
# choose k
k = 3
cluster_col = f"kmeans_k{k}"

poem_stats = (
    df_work
    .groupby(["poem_id", cluster_col])
    .size()
    .rename("n_fragments")
    .reset_index()
)

# total fragments per poem
poem_totals = (
    poem_stats
    .groupby("poem_id")["n_fragments"]
    .sum()
    .rename("total_fragments")
)

poem_stats = poem_stats.merge(poem_totals, on="poem_id")
poem_stats["share"] = poem_stats["n_fragments"] / poem_stats["total_fragments"]

# aggregate per poem
per_poem = (
    poem_stats
    .groupby("poem_id")
    .agg(
        n_clusters=(cluster_col, "nunique"),
        dominant_share=("share", "max")
    )
)


# cluster-count based
per_poem["cluster_count_category"] = pd.cut(
    per_poem["n_clusters"],
    bins=[0, 1, 2, per_poem["n_clusters"].max()],
    labels=["single-cluster", "mostly coherent", "split"],
    right=True
)

# dominance-strength based
per_poem["dominance_category"] = pd.cut(
    per_poem["dominant_share"],
    bins=[0.0, 0.5, 0.75, 1.0],
    labels=["diffuse", "moderately coherent", "strongly coherent"],
    include_lowest=True
)

# summary tables

cluster_count_summary = (
    per_poem["cluster_count_category"]
    .value_counts()
    .rename_axis("poem_behavior")
    .reset_index(name="n_poems")
)

dominance_summary = (
    per_poem["dominance_category"]
    .value_counts()
    .rename_axis("poem_behavior")
    .reset_index(name="n_poems")
)

print(f"{k} Clusters:")
print("How many clusters do poems spread across?")
display(cluster_count_summary)
print("How concentrated are a poem’s fragments in a single cluster?")
display(dominance_summary)


3 Clusters:
How many clusters do poems spread across?


,poem_behavior,n_poems
0,mostly coherent,29
1,single-cluster,18
2,split,15


How concentrated are a poem’s fragments in a single cluster?


,poem_behavior,n_poems
0,strongly coherent,31
1,moderately coherent,21
2,diffuse,10


In [ ]:
# choose k
k = 5
cluster_col = f"kmeans_k{k}"

poem_stats = (
    df_work
    .groupby(["poem_id", cluster_col])
    .size()
    .rename("n_fragments")
    .reset_index()
)

# total fragments per poem
poem_totals = (
    poem_stats
    .groupby("poem_id")["n_fragments"]
    .sum()
    .rename("total_fragments")
)

poem_stats = poem_stats.merge(poem_totals, on="poem_id")
poem_stats["share"] = poem_stats["n_fragments"] / poem_stats["total_fragments"]

# aggregate per poem
per_poem = (
    poem_stats
    .groupby("poem_id")
    .agg(
        n_clusters=(cluster_col, "nunique"),
        dominant_share=("share", "max")
    )
)


# cluster-count based
per_poem["cluster_count_category"] = pd.cut(
    per_poem["n_clusters"],
    bins=[0, 1, 2, per_poem["n_clusters"].max()],
    labels=["single-cluster", "mostly coherent", "split"],
    right=True
)

# dominance-strength based
per_poem["dominance_category"] = pd.cut(
    per_poem["dominant_share"],
    bins=[0.0, 0.5, 0.75, 1.0],
    labels=["diffuse", "moderately coherent", "strongly coherent"],
    include_lowest=True
)

# summary tables

cluster_count_summary = (
    per_poem["cluster_count_category"]
    .value_counts()
    .rename_axis("poem_behavior")
    .reset_index(name="n_poems")
)

dominance_summary = (
    per_poem["dominance_category"]
    .value_counts()
    .rename_axis("poem_behavior")
    .reset_index(name="n_poems")
)

print(f"{k} Clusters:")
print("How many clusters do poems spread across?")
display(cluster_count_summary)
print("How concentrated are a poem’s fragments in a single cluster?")
display(dominance_summary)


5 Clusters:
How many clusters do poems spread across?


,poem_behavior,n_poems
0,split,42
1,mostly coherent,17
2,single-cluster,3


How concentrated are a poem’s fragments in a single cluster?


,poem_behavior,n_poems
0,diffuse,37
1,moderately coherent,18
2,strongly coherent,7


In [ ]:
# choose k
k = 8
cluster_col = f"kmeans_k{k}"

poem_stats = (
    df_work
    .groupby(["poem_id", cluster_col])
    .size()
    .rename("n_fragments")
    .reset_index()
)

# total fragments per poem
poem_totals = (
    poem_stats
    .groupby("poem_id")["n_fragments"]
    .sum()
    .rename("total_fragments")
)

poem_stats = poem_stats.merge(poem_totals, on="poem_id")
poem_stats["share"] = poem_stats["n_fragments"] / poem_stats["total_fragments"]

# aggregate per poem
per_poem = (
    poem_stats
    .groupby("poem_id")
    .agg(
        n_clusters=(cluster_col, "nunique"),
        dominant_share=("share", "max")
    )
)


# cluster-count based
per_poem["cluster_count_category"] = pd.cut(
    per_poem["n_clusters"],
    bins=[0, 1, 2, per_poem["n_clusters"].max()],
    labels=["single-cluster", "mostly coherent", "split"],
    right=True
)

# dominance-strength based
per_poem["dominance_category"] = pd.cut(
    per_poem["dominant_share"],
    bins=[0.0, 0.5, 0.75, 1.0],
    labels=["diffuse", "moderately coherent", "strongly coherent"],
    include_lowest=True
)

# summary tables

cluster_count_summary = (
    per_poem["cluster_count_category"]
    .value_counts()
    .rename_axis("poem_behavior")
    .reset_index(name="n_poems")
)

dominance_summary = (
    per_poem["dominance_category"]
    .value_counts()
    .rename_axis("poem_behavior")
    .reset_index(name="n_poems")
)

print(f"{k} Clusters:")
print("How many clusters do poems spread across?")
display(cluster_count_summary)
print("How concentrated are a poem’s fragments in a single cluster?")
display(dominance_summary)


8 Clusters:
How many clusters do poems spread across?


,poem_behavior,n_poems
0,split,51
1,mostly coherent,10
2,single-cluster,1


How concentrated are a poem’s fragments in a single cluster?


,poem_behavior,n_poems
0,diffuse,41
1,moderately coherent,17
2,strongly coherent,4


**Clustering is not primarily driven by poem identity: at all resolutions, many poems distribute their fragments across multiple clusters**, indicating that fragment-level variation outweighs poem-level cohesion in the TF-IDF feature space. As the number of clusters increases, poem-level coherence further weakens, demonstrating that k-means partitions cut across poem boundaries rather than reflecting poem-level units.

### **2.4 By-Author Poem Analysis**

In [ ]:
# choose k
k = 3
cluster_col = f"kmeans_k{k}"

poem_stats = (
    df_work
    .groupby(["author", "poem_id", cluster_col])
    .size()
    .rename("n_fragments")
    .reset_index()
)

poem_totals = (
    poem_stats
    .groupby(["author", "poem_id"])["n_fragments"]
    .sum()
    .rename("total_fragments")
)

poem_stats = poem_stats.merge(poem_totals, on=["author", "poem_id"])
poem_stats["share"] = poem_stats["n_fragments"] / poem_stats["total_fragments"]

per_poem = (
    poem_stats
    .groupby(["author", "poem_id"])
    .agg(
        n_clusters=(cluster_col, "nunique"),
        dominant_share=("share", "max")
    )
    .reset_index()
)


per_poem["cluster_count_category"] = pd.cut(
    per_poem["n_clusters"],
    bins=[0, 1, 2, per_poem["n_clusters"].max()],
    labels=["single-cluster", "mostly coherent", "split"],
    right=True
)

per_poem["dominance_category"] = pd.cut(
    per_poem["dominant_share"],
    bins=[0.0, 0.5, 0.75, 1.0],
    labels=["diffuse", "moderately coherent", "strongly coherent"],
    include_lowest=True
)

# summaries

# How many clusters do poems spread across? (by author)
by_author_cluster_count = (
    per_poem
    .groupby(["author", "cluster_count_category"], observed=True)
    .size()
    .rename("n_poems")
    .reset_index()
    .sort_values(["author", "cluster_count_category"])
)

# How concentrated are poem fragments? (by author)
by_author_dominance = (
    per_poem
    .groupby(["author", "dominance_category"], observed=True)
    .size()
    .rename("n_poems")
    .reset_index()
    .sort_values(["author", "dominance_category"])
)

print(f"{k} Clusters:")
print("How do poems by each author spread across clusters?")
display(by_author_cluster_count)
print(f"{k} Clusters:")
print("How concentrated are poem fragments for each author?")
display(by_author_dominance)


3 Clusters:
How do poems by each author spread across clusters?


,author,cluster_count_category,n_poems
0,Billy Collins,single-cluster,3
1,Billy Collins,mostly coherent,6
2,Billy Collins,split,2
3,Louise Gluck,single-cluster,2
4,Louise Gluck,mostly coherent,5
5,Louise Gluck,split,6
6,Seamus Heaney,single-cluster,2
7,Seamus Heaney,mostly coherent,7
8,Seamus Heaney,split,3
9,Sharon Olds,single-cluster,6


3 Clusters:
How concentrated are poem fragments for each author?


,author,dominance_category,n_poems
0,Billy Collins,diffuse,2
1,Billy Collins,moderately coherent,2
2,Billy Collins,strongly coherent,7
3,Louise Gluck,diffuse,4
4,Louise Gluck,moderately coherent,7
5,Louise Gluck,strongly coherent,2
6,Seamus Heaney,diffuse,2
7,Seamus Heaney,moderately coherent,5
8,Seamus Heaney,strongly coherent,5
9,Sharon Olds,diffuse,2


In [ ]:
# choose k
k = 5
cluster_col = f"kmeans_k{k}"

poem_stats = (
    df_work
    .groupby(["author", "poem_id", cluster_col])
    .size()
    .rename("n_fragments")
    .reset_index()
)

poem_totals = (
    poem_stats
    .groupby(["author", "poem_id"])["n_fragments"]
    .sum()
    .rename("total_fragments")
)

poem_stats = poem_stats.merge(poem_totals, on=["author", "poem_id"])
poem_stats["share"] = poem_stats["n_fragments"] / poem_stats["total_fragments"]

per_poem = (
    poem_stats
    .groupby(["author", "poem_id"])
    .agg(
        n_clusters=(cluster_col, "nunique"),
        dominant_share=("share", "max")
    )
    .reset_index()
)


per_poem["cluster_count_category"] = pd.cut(
    per_poem["n_clusters"],
    bins=[0, 1, 2, per_poem["n_clusters"].max()],
    labels=["single-cluster", "mostly coherent", "split"],
    right=True
)

per_poem["dominance_category"] = pd.cut(
    per_poem["dominant_share"],
    bins=[0.0, 0.5, 0.75, 1.0],
    labels=["diffuse", "moderately coherent", "strongly coherent"],
    include_lowest=True
)

# summaries

# How many clusters do poems spread across? (by author)
by_author_cluster_count = (
    per_poem
    .groupby(["author", "cluster_count_category"], observed=True)
    .size()
    .rename("n_poems")
    .reset_index()
    .sort_values(["author", "cluster_count_category"])
)

# How concentrated are poem fragments? (by author)
by_author_dominance = (
    per_poem
    .groupby(["author", "dominance_category"], observed=True)
    .size()
    .rename("n_poems")
    .reset_index()
    .sort_values(["author", "dominance_category"])
)

print(f"{k} Clusters:")
print("How do poems by each author spread across clusters?")
display(by_author_cluster_count)
print(f"{k} Clusters:")
print("How concentrated are poem fragments for each author?")
display(by_author_dominance)


5 Clusters:
How do poems by each author spread across clusters?


,author,cluster_count_category,n_poems
0,Billy Collins,mostly coherent,3
1,Billy Collins,split,8
2,Louise Gluck,mostly coherent,3
3,Louise Gluck,split,10
4,Seamus Heaney,mostly coherent,3
5,Seamus Heaney,split,9
6,Sharon Olds,mostly coherent,4
7,Sharon Olds,split,10
8,Ted Hughes,single-cluster,3
9,Ted Hughes,mostly coherent,4


5 Clusters:
How concentrated are poem fragments for each author?


,author,dominance_category,n_poems
0,Billy Collins,diffuse,8
1,Billy Collins,moderately coherent,2
2,Billy Collins,strongly coherent,1
3,Louise Gluck,diffuse,9
4,Louise Gluck,moderately coherent,4
5,Seamus Heaney,diffuse,8
6,Seamus Heaney,moderately coherent,4
7,Sharon Olds,diffuse,9
8,Sharon Olds,moderately coherent,3
9,Sharon Olds,strongly coherent,2


In [ ]:
# choose k
k = 8
cluster_col = f"kmeans_k{k}"

poem_stats = (
    df_work
    .groupby(["author", "poem_id", cluster_col])
    .size()
    .rename("n_fragments")
    .reset_index()
)

poem_totals = (
    poem_stats
    .groupby(["author", "poem_id"])["n_fragments"]
    .sum()
    .rename("total_fragments")
)

poem_stats = poem_stats.merge(poem_totals, on=["author", "poem_id"])
poem_stats["share"] = poem_stats["n_fragments"] / poem_stats["total_fragments"]

per_poem = (
    poem_stats
    .groupby(["author", "poem_id"])
    .agg(
        n_clusters=(cluster_col, "nunique"),
        dominant_share=("share", "max")
    )
    .reset_index()
)


per_poem["cluster_count_category"] = pd.cut(
    per_poem["n_clusters"],
    bins=[0, 1, 2, per_poem["n_clusters"].max()],
    labels=["single-cluster", "mostly coherent", "split"],
    right=True
)

per_poem["dominance_category"] = pd.cut(
    per_poem["dominant_share"],
    bins=[0.0, 0.5, 0.75, 1.0],
    labels=["diffuse", "moderately coherent", "strongly coherent"],
    include_lowest=True
)

# summaries

# How many clusters do poems spread across? (by author)
by_author_cluster_count = (
    per_poem
    .groupby(["author", "cluster_count_category"], observed=True)
    .size()
    .rename("n_poems")
    .reset_index()
    .sort_values(["author", "cluster_count_category"])
)

# How concentrated are poem fragments? (by author)
by_author_dominance = (
    per_poem
    .groupby(["author", "dominance_category"], observed=True)
    .size()
    .rename("n_poems")
    .reset_index()
    .sort_values(["author", "dominance_category"])
)

print(f"{k} Clusters:")
print("How do poems by each author spread across clusters?")
display(by_author_cluster_count)
print(f"{k} Clusters:")
print("How concentrated are poem fragments for each author?")
display(by_author_dominance)


8 Clusters:
How do poems by each author spread across clusters?


,author,cluster_count_category,n_poems
0,Billy Collins,mostly coherent,2
1,Billy Collins,split,9
2,Louise Gluck,mostly coherent,2
3,Louise Gluck,split,11
4,Seamus Heaney,single-cluster,1
5,Seamus Heaney,split,11
6,Sharon Olds,mostly coherent,4
7,Sharon Olds,split,10
8,Ted Hughes,mostly coherent,2
9,Ted Hughes,split,10


8 Clusters:
How concentrated are poem fragments for each author?


,author,dominance_category,n_poems
0,Billy Collins,diffuse,7
1,Billy Collins,moderately coherent,3
2,Billy Collins,strongly coherent,1
3,Louise Gluck,diffuse,9
4,Louise Gluck,moderately coherent,4
5,Seamus Heaney,diffuse,9
6,Seamus Heaney,moderately coherent,2
7,Seamus Heaney,strongly coherent,1
8,Sharon Olds,diffuse,8
9,Sharon Olds,moderately coherent,5


Breaking poem-level behavior down by author shows the same general pattern across all poets. At k = 3, many poems (especially for Collins, Olds, and Hughes) remain mostly or strongly coherent, though splitting is already common. As the number of clusters increases, poems from every author spread across multiple clusters, with single-cluster poems becoming rare. **No author consistently retains poem-level coherence at higher resolutions**, suggesting that this behavior reflects the structure of the shared TF-IDF space rather than author-specific effects.

### **2.5 Notes in relation to earlier classification results**

The k-means analysis provides an unsupervised view of the TF-IDF feature space used in classification. Across all resolutions, clusters do not align cleanly with authors, reinforcing the conclusion that **the main limitations arise from the structure of the unigram TF-IDF space itself, and that further gains would likely require richer feature representations** rather than additional tuning within the same setup.

## **3. Structure-Revealing Clustering: HDBSCAN**

**Question:** Where is the space actually dense or coherent, and where is it diffuse?


This section uses HDBSCAN to examine the TF-IDF feature space. Unlike k-means, HDBSCAN does not force all fragments into clusters. Instead, it identifies denser regions while allowing ambiguous or weakly connected fragments to remain unassigned.

In this project, HDBSCAN serves as a complementary perspective to k-means. It is used to test whether the feature space contains any stable stylistic cores, and at what scale such structure appears. Noise is treated as an informative result rather than a failure of clustering.

HDBSCAN is chosen over standard DBSCAN because it can handle regions with different density levels.

**Two settings were chosen:**

* min_cluster_size = 12 (default min_samples): used to examine medium-scale dense regions without forcing very small clusters.

* min_cluster_size = 20, min_samples = 20: a stricter setting used to test whether any larger and more stable density cores exist.

In [ ]:
hdb_12 = hdbscan.HDBSCAN(
    min_cluster_size=12,
    metric="cosine"
)

labels_12 = hdb_12.fit_predict(X_tfidf)

# cluster sizes
cluster_sizes_12 = (
    pd.Series(labels_12)
      .value_counts()
      .sort_index()
)

cluster_sizes_12


,count
-1,414


In [ ]:
hdb_20 = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=20,
    metric="cosine"
)

labels_20 = hdb_20.fit_predict(X_tfidf)

# cluster sizes
cluster_sizes_20 = (
    pd.Series(labels_20)
      .value_counts()
      .sort_index()
)

cluster_sizes_20


,count
-1,414


In [ ]:
def summarize_hdbscan(labels):
    n_noise = np.sum(labels == -1)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    return {
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "n_total": len(labels)
    }

display(summarize_hdbscan(labels_12))
display(summarize_hdbscan(labels_20))

{'n_clusters': 0, 'n_noise': np.int64(414), 'n_total': 414}

{'n_clusters': 0, 'n_noise': np.int64(414), 'n_total': 414}

At reasonable scale definitions (12-20 fragments), HDBSCAN assigns all fragments to noise, indicating the absence of stable dense stylistic cores in the unigram TF-IDF space.

A much smaller cluster size (min_cluster_size = 5) is tested as a sanity check.

In [ ]:
hdb_diag = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=5,
    metric="cosine"
)

labels_diag = hdb_diag.fit_predict(X_tfidf)

# cluster sizes
cluster_sizes_diag = (
    pd.Series(labels_diag)
      .value_counts()
      .sort_index()
)

cluster_sizes_diag


,count
-1,387
0,22
1,5


In [ ]:
summarize_hdbscan(labels_diag)

{'n_clusters': 2, 'n_noise': np.int64(387), 'n_total': 414}

At this very small scale, HDBSCAN detects only tiny clusters, confirming that any density-based structure is limited to narrow, fragment-level pockets and does not extend to global stylistic organization.

## **4. Conclusions**

Taken together, the clustering analyses indicate that the unigram TF-IDF feature space is globally overlapping and weakly structured. k-means reveals only resolution-dependent partitions that vary with initialization and cut across poem and author boundaries, while HDBSCAN finds no stable dense cores at meaningful scales. Both perspectives support the same conclusion: stylistic signal in this representation is distributed and relational rather than organized into compact separable regions, helping explain the limitations observed earlier in supervised classification.
